# Find All Your Representatives by Zip Code

Looks up **state legislators** and **federal representatives** (senators + house rep) for any US zip code.

### Setup
You need one free API key:

| API | Sign-up | Cost |
|-----|---------|------|
| OpenStates | https://openstates.org/accounts/signup/ | Free |

In [ ]:
import requests
from pprint import pprint

In [ ]:
# ── Paste your API key here ──────────────────────────────────────────
OPENSTATES_API_KEY = "your-key-here"   # https://openstates.org/accounts/signup/ # ishani's api key

## 1 — Zip Code → Coordinates
Uses the free [Zippopotam.us](http://www.zippopotam.us/) API (no key needed).

In [ ]:
def zip_to_coords(zip_code):
    """Convert a US zip code to lat, lng, state, and place name."""
    res = requests.get(f"https://api.zippopotam.us/us/{zip_code}")
    res.raise_for_status()
    place = res.json()["places"][0]
    return {
        "lat": float(place["latitude"]),
        "lng": float(place["longitude"]),
        "state": place["state abbreviation"],
        "place_name": place["place name"],
    }

# Quick test
zip_to_coords("94110")

{'lat': 37.7509,
 'lng': -122.4153,
 'state': 'CA',
 'place_name': 'San Francisco'}

## 2 — Look Up All Representatives (OpenStates)

The OpenStates `people.geo` endpoint returns **both** state and federal legislators for a lat/lng.
We split them using `org_classification`:
- `"upper"` / `"lower"` → state senate / state house (assembly)
- anything else (e.g. `"legislature"`) → federal (US Senate / US House)

In [ ]:
def get_all_reps(zip_code):
    """
    Given a US zip code, return all state + federal representatives
    using Zippopotam (geocoding) and OpenStates (legislator lookup).
    """
    # Step 1: Geocode
    loc = zip_to_coords(zip_code)
    lat, lng = loc["lat"], loc["lng"]

    # Step 2: OpenStates lookup
    res = requests.get("https://v3.openstates.org/people.geo", params={
        "lat": lat,
        "lng": lng,
        "apikey": OPENSTATES_API_KEY,
        "include": ["offices", "links"],
    })
    res.raise_for_status()
    people = res.json().get("results", [])

    # Step 3: Parse and split into state vs federal
    state_legislators = []
    federal_legislators = []

    for person in people:
        role = person.get("current_role") or {}
        chamber_code = role.get("org_classification", "")

        # Collect office contact info
        offices = []
        for office in person.get("offices", []):
            offices.append({
                "type": office.get("classification"),
                "phone": office.get("voice"),
                "address": office.get("address"),
                "email": office.get("email"),
            })

        entry = {
            "name": person.get("name"),
            "party": person.get("party"),
            "title": role.get("title"),
            "district": role.get("district"),
            "image": person.get("image"),
            "offices": offices,
            "links": person.get("links", []),
        }

        if chamber_code in ("upper", "lower"):
            entry["chamber"] = "State Senate" if chamber_code == "upper" else "State Assembly/House"
            state_legislators.append(entry)
        else:
            entry["chamber"] = role.get("title", "Federal")
            federal_legislators.append(entry)

    return {
        "zip": zip_code,
        "location": loc,
        "state_legislators": state_legislators,
        "federal_legislators": federal_legislators,
    }

## 3 — Pretty Print

In [ ]:
def print_reps(result):
    """Nicely formatted output for a get_all_reps result."""
    loc = result["location"]
    print(f"📍 {loc['place_name']}, {loc['state']}  ({loc['lat']}, {loc['lng']})\n")

    def _print_person(r):
        print(f"  {r['title']} {r['name']} ({r['party']}) — {r.get('chamber', '')}")
        if r.get("district"):
            print(f"    District: {r['district']}")
        for o in r.get("offices", []):
            # Build a label like "Capitol Office" or "District Office — San Francisco"
            office_type = (o.get('type') or 'office').capitalize()
            # Extract city from address if available (typically "Street, City, ST ZIP")
            addr = o.get('address') or ''
            city = ''
            if addr:
                parts = [p.strip() for p in addr.split(',')]
                if len(parts) >= 2:
                    city = parts[-2]  # second-to-last part is usually the city
            label = f"{office_type} Office"
            if city:
                label += f" — {city}"
            print(f"    🏢 {label}")
            if o.get('phone'):
                print(f"       📞 {o['phone']}")
            if o.get('email'):
                print(f"       ✉  {o['email']}")
        # Website links (shown after offices)
        for link in r.get("links", []):
            if isinstance(link, dict):
                url = link.get("url", "")
                note = link.get("note", "").strip()
            else:
                url, note = link, ""
            if url:
                label = note if note else "Website"
                print(f"    🌐 {label}: {url}")
        print()

    print("── Federal Representatives ──")
    for r in result["federal_legislators"]:
        _print_person(r)

    print("── State Legislators ──")
    for r in result["state_legislators"]:
        _print_person(r)

In [ ]:
result2 = get_all_reps("95032")
print_reps(result2)

📍 Los Gatos, CA  (37.2417, -121.9554)

── Federal Representatives ──
── State Legislators ──
  Senator Adam Schiff (Democratic) — State Senate
    District: California
    🏢 Capitol Office — Washington
       📞 202-224-3841
    🏢 District Office — San Francisco
       📞 415-393-0707
    🏢 District Office — Fresno
       📞 559-485-7430
    🏢 District Office — Burbank
    🏢 District Office — San Diego
       📞 619-231-9712
    🏢 District Office — Los Angeles
    🌐 website: https://www.schiff.senate.gov
    🌐 Website: https://www.schiff.senate.gov/

  Senator Alex Padilla (Democratic) — State Senate
    District: California
    🏢 Capitol Office — Washington
       📞 202-224-3553
    🏢 District Office — Fresno
       📞 559-497-5109
    🏢 District Office — Los Angeles
       📞 310-231-4494
    🏢 District Office — San Francisco
       📞 415-981-9369
    🏢 District Office — Sacramento
       📞 916-448-2787
    🏢 District Office — San Diego
       📞 619-239-3884
    🌐 website: https://www.padi

In [ ]:
# Full raw data
pprint(result)

{'federal_legislators': [],
 'location': {'lat': 37.7509,
              'lng': -122.4153,
              'place_name': 'San Francisco',
              'state': 'CA'},
 'state_legislators': [{'chamber': 'State Senate',
                        'district': 'California',
                        'image': 'https://unitedstates.github.io/images/congress/450x550/S001150.jpg',
                        'links': [{'note': 'website',
                                   'url': 'https://www.schiff.senate.gov'},
                                  {'note': '',
                                   'url': 'https://www.schiff.senate.gov/'}],
                        'name': 'Adam Schiff',
                        'offices': [{'address': '112 Hart Senate Office '
                                                'Building, Washington, DC '
                                                '20510',
                                     'email': None,
                                     'phone': '202-224-3841',
        